# Drug Discovery Pipeline
## Quantum-Enhanced Multimodal Drug Discovery with Graph Neural Networks

### Architecture Overview
```
Multimodal Embeddings ──→ Transfer Learning ──→ Core Neural ──→ Model Engines ──→ Generation & Prediction
     from P1                Module + PTMs       Architectures     ├─ Graph Diffusion   ├─ De Novo SMILES
                                                    │             │  Engine (DiffRM)    │  Generation
Activity & ADMET ──→ PyTorch & PyG                  │             └─ Equivariant GNN   └─ Binding Affinity
    Scoring            Training Pipeline             │                                     Predictor
                                                     ▼
                                              Optimized Candidate Molecules
```

### Datasets
| Dataset | Records | Purpose |
|---------|---------|--------|
| drug_cleaned.csv | 76,543 | Docking scores + molecular descriptors |
| drug_filtered.csv | 637,292 | Filtered drug candidates |
| interactions_cleaned.csv | 750 | Drug-protein interactions |
| drug_protein_diagnosis.csv | 99 | Drug-protein binding with diagnosis |
| tox21_cleaned.csv | 7,831 | Toxicity assay data (12 endpoints) |
| clintox.csv | 1,484 | Clinical toxicity (FDA approval + CT toxicity) |

---
## 1. Setup & Installation

In [ ]:
# Install required packages (uncomment as needed)
# !pip install torch torch-geometric rdkit scikit-learn pandas numpy matplotlib seaborn tqdm

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

# PyTorch Geometric
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, global_add_pool
from torch_geometric.loader import DataLoader as PyGDataLoader

# RDKit for molecular processing
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw, DataStructs
from rdkit.Chem import rdMolDescriptors

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, accuracy_score, mean_squared_error,
                             classification_report, r2_score)

from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
print(f'PyG version: {torch_geometric.__version__}')

---
## 2. Data Loading & Exploration

In [ ]:
# Load all datasets
DATA_DIR = '.'

drug_cleaned = pd.read_csv(os.path.join(DATA_DIR, 'drug_cleaned.csv'))
drug_filtered = pd.read_csv(os.path.join(DATA_DIR, 'drug_filtered.csv'))
interactions = pd.read_csv(os.path.join(DATA_DIR, 'interactions_cleaned.csv'))
drug_protein = pd.read_csv(os.path.join(DATA_DIR, 'drug_protein_diagnosis.csv'))
tox21 = pd.read_csv(os.path.join(DATA_DIR, 'tox21_cleaned.csv'))
clintox = pd.read_csv(os.path.join(DATA_DIR, 'clintox.csv'))

print('=== Dataset Shapes ===')
for name, df in [('drug_cleaned', drug_cleaned), ('drug_filtered', drug_filtered),
                  ('interactions', interactions), ('drug_protein', drug_protein),
                  ('tox21', tox21), ('clintox', clintox)]:
    print(f'{name:30s} → {df.shape[0]:>8,} rows × {df.shape[1]:>3} cols')

In [ ]:
# Quick EDA
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Docking score distribution
axes[0, 0].hist(drug_cleaned['score'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Docking Scores (drug_cleaned)')
axes[0, 0].set_xlabel('Score')

# 2. MW distribution
axes[0, 1].hist(drug_cleaned['MW'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Molecular Weight Distribution')
axes[0, 1].axvline(x=500, color='red', linestyle='--', label='Lipinski MW<500')
axes[0, 1].legend()

# 3. Toxicity score distribution (tox21)
axes[0, 2].hist(tox21['toxicity_score'], bins=range(0, 14), color='darkred', edgecolor='black', alpha=0.7)
axes[0, 2].set_title('Tox21 Toxicity Scores')
axes[0, 2].set_xlabel('Number of toxic endpoints')

# 4. Binding affinity distribution
axes[1, 0].hist(drug_protein['binding_affinity'], bins=20, color='forestgreen', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Binding Affinities (drug_protein)')
axes[1, 0].set_xlabel('Binding Affinity')

# 5. Interaction types
interactions['interaction_type'].value_counts().plot.bar(ax=axes[1, 1], color='mediumpurple')
axes[1, 1].set_title('Interaction Types')
axes[1, 1].tick_params(axis='x', rotation=45)

# 6. ClinTox FDA approval vs toxicity
clintox_summary = clintox[['FDA_APPROVED', 'CT_TOX']].sum()
clintox_summary.plot.bar(ax=axes[1, 2], color=['green', 'red'])
axes[1, 2].set_title('ClinTox: FDA Approved vs CT Toxic')
axes[1, 2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

---
## 3. Multimodal Embeddings from P1
Generate molecular graph representations and fingerprint embeddings from SMILES.

In [ ]:
# ─── Atom and Bond Feature Extraction ───

ATOM_FEATURES = {
    'atomic_num': list(range(1, 119)),
    'degree': [0, 1, 2, 3, 4, 5],
    'formal_charge': [-2, -1, 0, 1, 2],
    'hybridization': [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3,
        Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2
    ],
    'num_hs': [0, 1, 2, 3, 4],
}

def one_hot_encode(value, choices):
    """One-hot encode a value given a list of possible choices."""
    encoding = [0] * (len(choices) + 1)  # +1 for unknown
    if value in choices:
        encoding[choices.index(value)] = 1
    else:
        encoding[-1] = 1
    return encoding


def get_atom_features(atom):
    """Extract atom-level features as a vector."""
    features = []
    features += one_hot_encode(atom.GetAtomicNum(), ATOM_FEATURES['atomic_num'])
    features += one_hot_encode(atom.GetDegree(), ATOM_FEATURES['degree'])
    features += one_hot_encode(atom.GetFormalCharge(), ATOM_FEATURES['formal_charge'])
    features += one_hot_encode(atom.GetHybridization(), ATOM_FEATURES['hybridization'])
    features += one_hot_encode(atom.GetTotalNumHs(), ATOM_FEATURES['num_hs'])
    features.append(1 if atom.GetIsAromatic() else 0)
    features.append(atom.GetMass() / 100.0)  # normalized mass
    return features


def get_bond_features(bond):
    """Extract bond-level features."""
    bt = bond.GetBondType()
    features = [
        bt == Chem.rdchem.BondType.SINGLE,
        bt == Chem.rdchem.BondType.DOUBLE,
        bt == Chem.rdchem.BondType.TRIPLE,
        bt == Chem.rdchem.BondType.AROMATIC,
        bond.GetIsConjugated(),
        bond.IsInRing()
    ]
    return [int(f) for f in features]


print(f'Atom feature dimension: {len(get_atom_features(Chem.MolFromSmiles("C").GetAtomWithIdx(0)))}')

In [ ]:
# ─── SMILES → PyG Graph Conversion ───

def smiles_to_graph(smiles, y=None):
    """Convert a SMILES string to a PyG Data object with atom/bond features."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Atom features
    atom_feats = [get_atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(atom_feats, dtype=torch.float)

    # Edge index and edge features
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index += [[i, j], [j, i]]
        bf = get_bond_features(bond)
        edge_attr += [bf, bf]

    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 6), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    data.smiles = smiles

    if y is not None:
        data.y = torch.tensor([y], dtype=torch.float)

    return data


# Test conversion
test_graph = smiles_to_graph('CCO')
print(f'Test graph: {test_graph}')
print(f'  Nodes: {test_graph.num_nodes}, Edges: {test_graph.num_edges}')
print(f'  Node feature dim: {test_graph.x.shape[1]}')

In [ ]:
# ─── Multimodal Molecular Embedding (Fingerprints + Descriptors) ───

def get_morgan_fingerprint(smiles, radius=2, n_bits=2048):
    """Generate Morgan fingerprint (ECFP) for a SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits)
    gen = AllChem.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fp = gen.GetFingerprintAsNumPy(mol)
    return fp.astype(np.float32)


def get_molecular_descriptors(smiles):
    """Compute physicochemical descriptors for a molecule."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(10)
    return np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol),
        Descriptors.FractionCSP3(mol),
        Descriptors.NumAromaticRings(mol),
        Descriptors.HeavyAtomCount(mol)
    ], dtype=np.float32)


def get_multimodal_embedding(smiles, fp_bits=2048):
    """Concatenate fingerprint + descriptors into a single embedding vector."""
    fp = get_morgan_fingerprint(smiles, n_bits=fp_bits)
    desc = get_molecular_descriptors(smiles)
    return np.concatenate([fp, desc])


# Demo
emb = get_multimodal_embedding('CCO')
print(f'Multimodal embedding dimension: {len(emb)} (2048 FP + 10 descriptors)')
# Demo

---
## 4. Transfer Learning Module → Pre-trained Foundation Models
A transfer learning encoder that learns latent molecular representations. Pre-trained on large-scale molecular data and fine-tuned for downstream tasks.

In [ ]:
# ─── Pre-trained Molecular Encoder (Foundation Model) ───

class MolecularFoundationEncoder(nn.Module):
    """Foundation model: encodes multimodal molecular embeddings into a
    latent space. Can be pre-trained then frozen for transfer learning."""

    def __init__(self, input_dim=2058, hidden_dim=512, latent_dim=256, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, latent_dim),
        )
        # Decoder for pre-training (reconstruction)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return z, recon


class TransferLearningModule(nn.Module):
    """Wraps the foundation encoder w/ a fine-tuning head for any downstream task."""

    def __init__(self, foundation_encoder, latent_dim=256, output_dim=1, freeze_encoder=True):
        super().__init__()
        self.foundation = foundation_encoder
        if freeze_encoder:
            for param in self.foundation.encoder.parameters():
                param.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        z = self.foundation.encode(x)
        return self.head(z)


print('Foundation Encoder parameters:', sum(p.numel() for p in MolecularFoundationEncoder().parameters()))

In [ ]:
# ─── Pre-train the Foundation Encoder (Autoencoder on drug SMILES) ───

class MolEmbeddingDataset(Dataset):
    def __init__(self, smiles_list, fp_bits=2048):
        self.embeddings = []
        for smi in tqdm(smiles_list, desc='Building embeddings'):
            emb = get_multimodal_embedding(smi, fp_bits=fp_bits)
            self.embeddings.append(emb)
        self.embeddings = np.array(self.embeddings)
        # Normalize descriptors (last 10 features)
        self.scaler = StandardScaler()
        self.embeddings[:, -10:] = self.scaler.fit_transform(self.embeddings[:, -10:])

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return torch.tensor(self.embeddings[idx], dtype=torch.float)


# Use a sample of drug_cleaned for pre-training
pretrain_smiles = drug_cleaned['SMILES'].dropna().sample(n=min(10000, len(drug_cleaned)), random_state=42).tolist()
pretrain_dataset = MolEmbeddingDataset(pretrain_smiles)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=256, shuffle=True)

print(f'Pre-training on {len(pretrain_dataset)} molecules')

In [ ]:
# ─── Pre-training Loop ───

foundation_model = MolecularFoundationEncoder(input_dim=2058, latent_dim=256).to(DEVICE)
optimizer_pt = torch.optim.Adam(foundation_model.parameters(), lr=1e-3)

PRETRAIN_EPOCHS = 30
pretrain_losses = []

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    foundation_model.train()
    epoch_loss = 0
    for batch in pretrain_loader:
        batch = batch.to(DEVICE)
        z, recon = foundation_model(batch)
        loss = F.mse_loss(recon, batch)
        optimizer_pt.zero_grad()
        loss.backward()
        optimizer_pt.step()
        epoch_loss += loss.item() * batch.size(0)
    epoch_loss /= len(pretrain_dataset)
    pretrain_losses.append(epoch_loss)
    if epoch % 5 == 0:
        print(f'  Epoch {epoch:3d}/{PRETRAIN_EPOCHS} │ Recon Loss: {epoch_loss:.6f}')

plt.figure(figsize=(8, 4))
plt.plot(pretrain_losses, color='royalblue')
plt.xlabel('Epoch'); plt.ylabel('Reconstruction Loss')
plt.title('Foundation Model Pre-training')
plt.grid(True, alpha=0.3)
plt.show()

print('Foundation model pre-training complete.')

---
## 5. Activity & ADMET Scoring Pipeline
Toxicity prediction using Tox21 + ClinTox data. This feeds into the training pipeline for downstream molecule optimization.

In [ ]:
# ─── Build Tox21 Graph Dataset ───

TOX_ENDPOINTS = [
    'androgen_receptor', 'androgen_receptor_lbd', 'aryl_hydrocarbon_receptor',
    'aromatase', 'estrogen_receptor', 'estrogen_receptor_lbd', 'ppar_gamma',
    'antioxidant_response', 'dna_damage', 'heat_shock',
    'mitochondrial_membrane', 'tumor_suppressor_p53'
]

def build_tox_graph_dataset(df, smiles_col, label_col, max_samples=5000):
    """Build graph dataset for toxicity prediction."""
    graphs = []
    df_sample = df.dropna(subset=[smiles_col, label_col]).head(max_samples)
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc='Building tox graphs'):
        g = smiles_to_graph(row[smiles_col], y=float(row[label_col]))
        if g is not None:
            graphs.append(g)
    return graphs


# Build Tox21 dataset (binary: is_toxic)
tox21_graphs = build_tox_graph_dataset(tox21, 'drug_smiles', 'is_toxic', max_samples=5000)
print(f'Tox21 graphs built: {len(tox21_graphs)}')

# Train/val/test split
train_graphs, test_graphs = train_test_split(tox21_graphs, test_size=0.2, random_state=42)
train_graphs, val_graphs = train_test_split(train_graphs, test_size=0.15, random_state=42)

print(f'Train: {len(train_graphs)}, Val: {len(val_graphs)}, Test: {len(test_graphs)}')

tox_train_loader = PyGDataLoader(train_graphs, batch_size=64, shuffle=True)
tox_val_loader = PyGDataLoader(val_graphs, batch_size=64)
tox_test_loader = PyGDataLoader(test_graphs, batch_size=64)

In [ ]:
# ─── ClinTox Dataset ───

clintox_graphs = build_tox_graph_dataset(clintox, 'smiles', 'CT_TOX', max_samples=1484)
print(f'ClinTox graphs built: {len(clintox_graphs)}')

clintox_train, clintox_test = train_test_split(clintox_graphs, test_size=0.2, random_state=42)
clintox_train_loader = PyGDataLoader(clintox_train, batch_size=64, shuffle=True)
clintox_test_loader = PyGDataLoader(clintox_test, batch_size=64)

---
## 6. Core Neural Architectures
GNN backbone shared by the Diffusion and Equivariant paths.

In [ ]:
# ─── Core GNN Backbone ───

class CoreGNNBackbone(nn.Module):
    """Shared GNN backbone that processes molecular graphs.
    Outputs node embeddings and a graph-level embedding."""

    def __init__(self, node_feat_dim, hidden_dim=256, num_layers=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        self.dropout = dropout

    def forward(self, x, edge_index, batch=None):
        x = self.input_proj(x)
        for conv, bn in zip(self.convs, self.bns):
            x_res = x
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + x_res  # residual connection

        # Graph-level readout
        if batch is not None:
            graph_emb = global_mean_pool(x, batch)
        else:
            graph_emb = x.mean(dim=0, keepdim=True)

        return x, graph_emb  # node embeddings, graph embedding


NODE_FEAT_DIM = tox21_graphs[0].x.shape[1]
print(f'Node feature dimension: {NODE_FEAT_DIM}')

---
## 7. Model Engines

### 7a. Diffusion Path → Graph Diffusion Engine (DiffRM)
Denoising diffusion model on molecular graphs for **de novo molecule generation**.

In [ ]:
# ─── Graph Diffusion Engine (DiffRM) ───

class GraphDiffusionEngine(nn.Module):
    """Simplified graph diffusion model for molecular generation.
    Learns to denoise molecular graph embeddings via a diffusion process."""

    def __init__(self, node_feat_dim, hidden_dim=256, latent_dim=128, num_steps=100):
        super().__init__()
        self.num_steps = num_steps
        self.latent_dim = latent_dim

        # GNN encoder
        self.backbone = CoreGNNBackbone(node_feat_dim, hidden_dim)

        # Project graph embedding to latent space
        self.proj = nn.Linear(hidden_dim, latent_dim)

        # Time embedding
        self.time_embed = nn.Sequential(
            nn.Linear(1, 64),
            nn.SiLU(),
            nn.Linear(64, latent_dim)
        )

        # Noise prediction network
        self.noise_pred = nn.Sequential(
            nn.Linear(latent_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim)
        )

        # Noise schedule (linear beta schedule)
        betas = torch.linspace(1e-4, 0.02, num_steps)
        alphas = 1.0 - betas
        alpha_bar = torch.cumprod(alphas, dim=0)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas', alphas)
        self.register_buffer('alpha_bar', alpha_bar)

    def encode_graph(self, data):
        """Encode a molecular graph batch to latent vectors."""
        _, graph_emb = self.backbone(data.x, data.edge_index, data.batch)
        return self.proj(graph_emb)

    def forward_diffusion(self, z0, t):
        """Add noise to latent z0 at timestep t."""
        noise = torch.randn_like(z0)
        alpha_bar_t = self.alpha_bar[t].unsqueeze(-1)
        zt = torch.sqrt(alpha_bar_t) * z0 + torch.sqrt(1 - alpha_bar_t) * noise
        return zt, noise

    def predict_noise(self, zt, t):
        """Predict the noise added at timestep t."""
        t_emb = self.time_embed(t.float().unsqueeze(-1) / self.num_steps)
        combined = torch.cat([zt, t_emb], dim=-1)
        return self.noise_pred(combined)

    def training_loss(self, data):
        """Compute diffusion training loss."""
        z0 = self.encode_graph(data)
        t = torch.randint(0, self.num_steps, (z0.size(0),), device=z0.device)
        zt, noise = self.forward_diffusion(z0, t)
        noise_pred = self.predict_noise(zt, t)
        return F.mse_loss(noise_pred, noise)

    @torch.no_grad()
    def sample(self, num_samples=10):
        """Generate new molecular latent vectors via reverse diffusion."""
        self.eval()
        z = torch.randn(num_samples, self.latent_dim, device=next(self.parameters()).device)
        for t_idx in reversed(range(self.num_steps)):
            t = torch.full((num_samples,), t_idx, device=z.device, dtype=torch.long)
            noise_pred = self.predict_noise(z, t)
            alpha_t = self.alphas[t_idx]
            alpha_bar_t = self.alpha_bar[t_idx]
            z = (1 / torch.sqrt(alpha_t)) * (
                z - (self.betas[t_idx] / torch.sqrt(1 - alpha_bar_t)) * noise_pred
            )
            if t_idx > 0:
                z += torch.sqrt(self.betas[t_idx]) * torch.randn_like(z)
        return z  # denoised latent vectors


diffusion_engine = GraphDiffusionEngine(NODE_FEAT_DIM).to(DEVICE)
print(f'Diffusion Engine parameters: {sum(p.numel() for p in diffusion_engine.parameters()):,}')

### 7b. Equivariant Path → Equivariant Graph Neural Network (EGNN)
E(n)-equivariant GNN with attention mechanisms for **binding affinity prediction**.

In [ ]:
# ─── Equivariant Graph Neural Network Layer ───

class EGNNLayer(nn.Module):
    """E(n) Equivariant Graph Neural Network layer.
    Updates node features using message passing with coordinate equivariance."""

    def __init__(self, hidden_dim, edge_feat_dim=0):
        super().__init__()
        # Message MLP
        self.msg_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1 + edge_feat_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        # Node update MLP
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        # Attention
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, h, edge_index, edge_attr=None):
        row, col = edge_index
        # Distance feature (using feature-space distance as proxy for coordinates)
        dist = torch.sum((h[row] - h[col]) ** 2, dim=-1, keepdim=True)

        # Build message input
        msg_input = torch.cat([h[row], h[col], dist], dim=-1)
        if edge_attr is not None:
            msg_input = torch.cat([msg_input, edge_attr], dim=-1)

        msg = self.msg_mlp(msg_input)
        attn = self.attention(msg)
        msg = msg * attn  # attention-weighted messages

        # Aggregate messages
        agg = torch.zeros_like(h)
        agg.index_add_(0, col, msg)

        # Update nodes
        h_new = self.node_mlp(torch.cat([h, agg], dim=-1))
        return h + h_new  # residual


class EquivariantGNN(nn.Module):
    """Full Equivariant GNN for binding affinity prediction."""

    def __init__(self, node_feat_dim, hidden_dim=256, num_layers=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)
        self.layers = nn.ModuleList([
            EGNNLayer(hidden_dim) for _ in range(num_layers)
        ])
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(num_layers)
        ])

        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)  # binding affinity
        )
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        h = self.input_proj(x)

        for layer, ln in zip(self.layers, self.layer_norms):
            h = layer(h, edge_index)
            h = ln(h)
            h = F.dropout(h, p=self.dropout, training=self.training)

        # Graph-level readout via attention pooling
        graph_emb = global_mean_pool(h, batch)
        return self.predictor(graph_emb)


egnn_model = EquivariantGNN(NODE_FEAT_DIM).to(DEVICE)
print(f'EGNN parameters: {sum(p.numel() for p in egnn_model.parameters()):,}')

---
## 8. De Novo SMILES Generation (Diffusion Path Output)
A SMILES decoder that maps diffusion latent vectors to valid molecular SMILES strings.

In [ ]:
# ─── SMILES Tokenizer ───

import re

SMILES_CHARS = [
    '<pad>', '<sos>', '<eos>',
    'C', 'c', 'N', 'n', 'O', 'o', 'S', 's', 'F', 'I',
    'Cl', 'Br', 'P', 'B', 'Si',
    '(', ')', '[', ']', '=', '#', '+', '-',
    '1', '2', '3', '4', '5', '6', '7', '8', '9',
    '@', '.', '/', '\\', '%', ':', 'H'
]

CHAR2IDX = {c: i for i, c in enumerate(SMILES_CHARS)}
IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}
VOCAB_SIZE = len(SMILES_CHARS)

SMILES_REGEX = re.compile(r'Cl|Br|Si|@@?|\[.*?\]|%\d{2}|.')

def tokenize_smiles(smiles, max_len=120):
    tokens = [CHAR2IDX.get('<sos>')]
    for token in SMILES_REGEX.findall(smiles):
        if token in CHAR2IDX:
            tokens.append(CHAR2IDX[token])
        else:
            for ch in token:
                tokens.append(CHAR2IDX.get(ch, CHAR2IDX['<pad>']))
    tokens.append(CHAR2IDX.get('<eos>'))
    # Pad/truncate
    if len(tokens) > max_len:
        tokens = tokens[:max_len]
    else:
        tokens += [CHAR2IDX['<pad>']] * (max_len - len(tokens))
    return tokens


def decode_tokens(token_ids):
    smiles = ''
    for t in token_ids:
        ch = IDX2CHAR.get(t, '')
        if ch == '<eos>':
            break
        if ch not in ('<pad>', '<sos>'):
            smiles += ch
    return smiles


print(f'SMILES vocabulary size: {VOCAB_SIZE}')
print(f'Tokenized "CCO": {tokenize_smiles("CCO", max_len=10)}')
print(f'Decoded back: "{decode_tokens(tokenize_smiles("CCO", max_len=10))}"')

In [ ]:
# ─── Latent-to-SMILES Decoder ───

class SMILESDecoder(nn.Module):
    """Autoregressive SMILES decoder from latent vectors."""

    def __init__(self, latent_dim=128, hidden_dim=512, vocab_size=VOCAB_SIZE, max_len=120):
        super().__init__()
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.latent_proj = nn.Linear(latent_dim, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.output_proj = nn.Linear(hidden_dim, vocab_size)

    def forward(self, z, target_tokens=None, teacher_forcing_ratio=0.5):
        """Decode latent vector z to SMILES token sequence."""
        batch_size = z.size(0)
        h = self.latent_proj(z).unsqueeze(0).repeat(2, 1, 1)  # GRU hidden state

        # Start with <sos>
        input_token = torch.full((batch_size, 1), CHAR2IDX['<sos>'],
                                 dtype=torch.long, device=z.device)
        outputs = []

        for t in range(self.max_len):
            emb = self.embedding(input_token)
            out, h = self.gru(emb, h)
            logits = self.output_proj(out)
            outputs.append(logits)

            if target_tokens is not None and np.random.random() < teacher_forcing_ratio:
                input_token = target_tokens[:, t:t+1]
            else:
                input_token = logits.argmax(dim=-1)

        return torch.cat(outputs, dim=1)  # (batch, max_len, vocab_size)

    @torch.no_grad()
    def generate(self, z, temperature=0.8, top_k=15):
        """Generate SMILES from latent vectors with top-k temperature sampling."""
        self.eval()
        batch_size = z.size(0)
        h = self.latent_proj(z).unsqueeze(0).repeat(2, 1, 1)
        input_token = torch.full((batch_size, 1), CHAR2IDX['<sos>'],
                                 dtype=torch.long, device=z.device)
        generated = []

        for _ in range(self.max_len):
            emb = self.embedding(input_token)
            out, h = self.gru(emb, h)
            logits = self.output_proj(out[:, -1, :]) / temperature

            # Top-k filtering: zero out all logits below top-k
            if top_k > 0 and top_k < logits.size(-1):
                top_vals, top_idx = torch.topk(logits, top_k, dim=-1)
                probs = F.softmax(top_vals, dim=-1)
                sampled = torch.multinomial(probs, 1)
                next_token = top_idx.gather(-1, sampled)
            else:
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, 1)

            generated.append(next_token)
            input_token = next_token

        generated = torch.cat(generated, dim=1)
        smiles_list = [decode_tokens(seq.cpu().numpy()) for seq in generated]
        return smiles_list


smiles_decoder = SMILESDecoder(latent_dim=128).to(DEVICE)
print(f'SMILES Decoder parameters: {sum(p.numel() for p in smiles_decoder.parameters()):,}')

---
## 9. Binding Affinity Predictor (Equivariant Path Output)
Predicts drug-protein binding affinity using EGNN with attention mechanisms.

In [ ]:
# ─── Build Binding Affinity Dataset ───

def build_affinity_dataset(df):
    """Build graph dataset for binding affinity prediction."""
    graphs = []
    for _, row in df.iterrows():
        g = smiles_to_graph(row['drug_smiles'], y=float(row['binding_affinity']))
        if g is not None:
            g.drug_name = row['drug_name']
            g.protein_name = row['protein_name']
            g.diagnosis = row['diagnosis']
            graphs.append(g)
    return graphs


affinity_graphs = build_affinity_dataset(drug_protein)
print(f'Binding affinity graphs: {len(affinity_graphs)}')

aff_train, aff_test = train_test_split(affinity_graphs, test_size=0.2, random_state=42)
aff_train_loader = PyGDataLoader(aff_train, batch_size=16, shuffle=True)
aff_test_loader = PyGDataLoader(aff_test, batch_size=16)

---
## 10. PyTorch & PyG Training Pipeline
Unified training loop for all model components.

In [ ]:
# ─── Toxicity Classifier (ADMET Scoring) ───

class ToxicityClassifier(nn.Module):
    """GNN-based toxicity classifier using the core backbone."""

    def __init__(self, node_feat_dim, hidden_dim=256, num_layers=4):
        super().__init__()
        self.backbone = CoreGNNBackbone(node_feat_dim, hidden_dim, num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, data):
        _, graph_emb = self.backbone(data.x, data.edge_index, data.batch)
        return self.classifier(graph_emb)


tox_model = ToxicityClassifier(NODE_FEAT_DIM).to(DEVICE)
print(f'Toxicity Classifier parameters: {sum(p.numel() for p in tox_model.parameters()):,}')

In [ ]:
# ─── Train Toxicity Classifier (Tox21) ───

tox_optimizer = torch.optim.Adam(tox_model.parameters(), lr=1e-3, weight_decay=1e-5)
tox_scheduler = CosineAnnealingLR(tox_optimizer, T_max=30)
bce_loss = nn.BCEWithLogitsLoss()

TOX_EPOCHS = 30
best_val_auc = 0
train_losses, val_aucs = [], []

for epoch in range(1, TOX_EPOCHS + 1):
    # Train
    tox_model.train()
    total_loss = 0
    for batch in tox_train_loader:
        batch = batch.to(DEVICE)
        logits = tox_model(batch).squeeze(-1)
        loss = bce_loss(logits, batch.y.squeeze(-1))
        tox_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tox_model.parameters(), 1.0)
        tox_optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    tox_scheduler.step()
    train_losses.append(total_loss / len(train_graphs))

    # Validate
    tox_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in tox_val_loader:
            batch = batch.to(DEVICE)
            logits = tox_model(batch).squeeze(-1)
            preds.append(torch.sigmoid(logits).cpu())
            labels.append(batch.y.squeeze(-1).cpu())
    preds = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    auc = roc_auc_score(labels, preds)
    val_aucs.append(auc)

    if auc > best_val_auc:
        best_val_auc = auc
        torch.save(tox_model.state_dict(), 'best_tox_model.pt')

    if epoch % 5 == 0:
        print(f'  Epoch {epoch:3d}/{TOX_EPOCHS} │ Loss: {train_losses[-1]:.4f} │ Val AUC: {auc:.4f}')

print(f'\nBest Validation AUC: {best_val_auc:.4f}')

In [ ]:
# ─── Evaluate Toxicity Classifier on Test Set ───

tox_model.load_state_dict(torch.load('best_tox_model.pt', weights_only=True))
tox_model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for batch in tox_test_loader:
        batch = batch.to(DEVICE)
        logits = tox_model(batch).squeeze(-1)
        test_preds.append(torch.sigmoid(logits).cpu())
        test_labels.append(batch.y.squeeze(-1).cpu())

test_preds = torch.cat(test_preds).numpy()
test_labels = torch.cat(test_labels).numpy()

test_auc = roc_auc_score(test_labels, test_preds)
test_acc = accuracy_score(test_labels, (test_preds > 0.5).astype(int))
print(f'Test AUC: {test_auc:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')
print()
print(classification_report(test_labels, (test_preds > 0.5).astype(int),
                            target_names=['Non-toxic', 'Toxic']))

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses, color='red', label='Train Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Toxicity Model Training Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(val_aucs, color='blue', label='Val AUC')
ax2.axhline(y=test_auc, color='green', linestyle='--', label=f'Test AUC={test_auc:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('AUC')
ax2.set_title('Toxicity Model Validation AUC'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Train Diffusion Engine (Graph Diffusion for Molecule Generation) ───

diff_optimizer = torch.optim.Adam(diffusion_engine.parameters(), lr=5e-4)
diff_scheduler = CosineAnnealingLR(diff_optimizer, T_max=50)

# Use tox21 non-toxic molecules for training generation model
nontoxic_smiles = tox21[tox21['is_toxic'] == 0]['drug_smiles'].tolist()
gen_graphs = []
for smi in tqdm(nontoxic_smiles[:3000], desc='Building generation graphs'):
    g = smiles_to_graph(smi)
    if g is not None:
        gen_graphs.append(g)

gen_loader = PyGDataLoader(gen_graphs, batch_size=64, shuffle=True)
print(f'Generation training graphs: {len(gen_graphs)}')

DIFF_EPOCHS = 50
diff_losses = []

for epoch in range(1, DIFF_EPOCHS + 1):
    diffusion_engine.train()
    total_loss = 0
    for batch in gen_loader:
        batch = batch.to(DEVICE)
        loss = diffusion_engine.training_loss(batch)
        diff_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(diffusion_engine.parameters(), 1.0)
        diff_optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    diff_scheduler.step()
    diff_losses.append(total_loss / len(gen_graphs))

    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d}/{DIFF_EPOCHS} │ Diffusion Loss: {diff_losses[-1]:.6f}')

plt.figure(figsize=(8, 4))
plt.plot(diff_losses, color='purple')
plt.xlabel('Epoch'); plt.ylabel('Diffusion Loss')
plt.title('Graph Diffusion Engine Training')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ─── Train SMILES Decoder (paired with Diffusion Engine latents) ───

from torch.utils.data import TensorDataset

# Pre-compute latent vectors by encoding graphs through the diffusion engine
diffusion_engine.eval()
paired_smiles = [g.smiles for g in gen_graphs[:2000]]
pre_loader = PyGDataLoader(gen_graphs[:2000], batch_size=64, shuffle=False)

all_z = []
with torch.no_grad():
    for batch in pre_loader:
        batch = batch.to(DEVICE)
        z = diffusion_engine.encode_graph(batch)
        all_z.append(z.cpu())
gen_z = torch.cat(all_z, dim=0)
print(f'Pre-computed latent vectors: {gen_z.shape}')

# Tokenize SMILES
all_tokens = torch.stack([
    torch.tensor(tokenize_smiles(s, max_len=120), dtype=torch.long)
    for s in paired_smiles
])

# Create paired (z, tokens) dataset
paired_dataset = TensorDataset(gen_z, all_tokens)
paired_loader = DataLoader(paired_dataset, batch_size=64, shuffle=True)

dec_optimizer = torch.optim.Adam(smiles_decoder.parameters(), lr=1e-3)
dec_scheduler = CosineAnnealingLR(dec_optimizer, T_max=50)

DEC_EPOCHS = 50
dec_losses = []

for epoch in range(1, DEC_EPOCHS + 1):
    smiles_decoder.train()
    # Scheduled sampling: teacher forcing decays from 1.0 → 0.0
    tf_ratio = max(0.0, 1.0 - epoch / DEC_EPOCHS)
    total_loss = 0
    for z_batch, tokens_batch in paired_loader:
        z_batch = z_batch.to(DEVICE)
        tokens_batch = tokens_batch.to(DEVICE)
        logits = smiles_decoder(z_batch, target_tokens=tokens_batch,
                                teacher_forcing_ratio=tf_ratio)
        loss = F.cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            tokens_batch.view(-1),
            ignore_index=CHAR2IDX['<pad>']
        )
        dec_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(smiles_decoder.parameters(), 1.0)
        dec_optimizer.step()
        total_loss += loss.item() * z_batch.size(0)
    dec_scheduler.step()
    dec_losses.append(total_loss / len(paired_dataset))

    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d}/{DEC_EPOCHS} │ Decoder Loss: {dec_losses[-1]:.4f} │ TF Ratio: {tf_ratio:.2f}')

plt.figure(figsize=(8, 4))
plt.plot(dec_losses, color='teal')
plt.xlabel('Epoch'); plt.ylabel('Decoder Loss')
plt.title('SMILES Decoder Training (Latent-Conditioned)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ─── Train Binding Affinity Predictor (EGNN) ───

aff_optimizer = torch.optim.Adam(egnn_model.parameters(), lr=1e-3, weight_decay=1e-5)
aff_scheduler = CosineAnnealingLR(aff_optimizer, T_max=50)

AFF_EPOCHS = 50
aff_train_losses, aff_val_losses = [], []

for epoch in range(1, AFF_EPOCHS + 1):
    egnn_model.train()
    total_loss = 0
    n_samples = 0
    for batch in aff_train_loader:
        batch = batch.to(DEVICE)
        pred = egnn_model(batch.x, batch.edge_index, batch.batch).squeeze(-1)
        loss = F.mse_loss(pred, batch.y.squeeze(-1))
        aff_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(egnn_model.parameters(), 1.0)
        aff_optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        n_samples += batch.num_graphs
    aff_scheduler.step()
    aff_train_losses.append(total_loss / n_samples)

    # Evaluate
    egnn_model.eval()
    val_loss = 0
    n_val = 0
    with torch.no_grad():
        for batch in aff_test_loader:
            batch = batch.to(DEVICE)
            pred = egnn_model(batch.x, batch.edge_index, batch.batch).squeeze(-1)
            val_loss += F.mse_loss(pred, batch.y.squeeze(-1)).item() * batch.num_graphs
            n_val += batch.num_graphs
    aff_val_losses.append(val_loss / n_val)

    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d}/{AFF_EPOCHS} │ Train MSE: {aff_train_losses[-1]:.4f} │ Val MSE: {aff_val_losses[-1]:.4f}')

# Plot
plt.figure(figsize=(8, 4))
plt.plot(aff_train_losses, label='Train MSE', color='navy')
plt.plot(aff_val_losses, label='Val MSE', color='orange')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Binding Affinity Predictor (EGNN) Training')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ─── Evaluate Binding Affinity Predictor ───

egnn_model.eval()
all_preds, all_labels = [], []
drug_names, protein_names, diagnoses = [], [], []

with torch.no_grad():
    for batch in aff_test_loader:
        batch = batch.to(DEVICE)
        pred = egnn_model(batch.x, batch.edge_index, batch.batch).squeeze(-1)
        all_preds.append(pred.cpu())
        all_labels.append(batch.y.squeeze(-1).cpu())

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

rmse = np.sqrt(mean_squared_error(all_labels, all_preds))
r2 = r2_score(all_labels, all_preds)
print(f'Binding Affinity Prediction — RMSE: {rmse:.4f}, R²: {r2:.4f}')

plt.figure(figsize=(6, 6))
plt.scatter(all_labels, all_preds, alpha=0.6, edgecolors='black', linewidths=0.5)
mn, mx = min(all_labels.min(), all_preds.min()) - 0.5, max(all_labels.max(), all_preds.max()) + 0.5
plt.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5)
plt.xlabel('True Binding Affinity')
plt.ylabel('Predicted Binding Affinity')
plt.title(f'Binding Affinity: Predicted vs True (R²={r2:.3f})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 11. De Novo SMILES Generation (Diffusion → Decoder)
Generate novel molecules using the trained diffusion engine + SMILES decoder.

In [ ]:
# ─── Generate Novel Molecules ───

def repair_smiles(smiles):
    """Attempt to fix common SMILES syntax errors (bracket/paren balancing)."""
    # 1. Balance parentheses — drop extra ')' and close unclosed '('
    depth = 0
    result = []
    for c in smiles:
        if c == '(':
            depth += 1
            result.append(c)
        elif c == ')':
            if depth > 0:
                depth -= 1
                result.append(c)
        else:
            result.append(c)
    result.extend([')'] * depth)

    # 2. Balance square brackets — drop extra ']' and close unclosed '['
    smiles = ''.join(result)
    depth = 0
    result = []
    for c in smiles:
        if c == '[':
            depth += 1
            result.append(c)
        elif c == ']':
            if depth > 0:
                depth -= 1
                result.append(c)
        else:
            result.append(c)
    result.extend([']'] * depth)
    return ''.join(result)


NUM_GENERATE = 100

# Sample latent vectors from diffusion engine
diffusion_engine.eval()
latent_vectors = diffusion_engine.sample(num_samples=NUM_GENERATE)

# Decode to SMILES with top-k sampling and lower temperature
generated_smiles = smiles_decoder.generate(latent_vectors, temperature=0.7, top_k=15)

# Validate generated SMILES (with repair attempt for invalid ones)
valid_smiles = []
repaired_count = 0
for smi in generated_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        valid_smiles.append(Chem.MolToSmiles(mol))
    else:
        repaired = repair_smiles(smi)
        mol = Chem.MolFromSmiles(repaired)
        if mol is not None:
            valid_smiles.append(Chem.MolToSmiles(mol))
            repaired_count += 1

unique_smiles = list(set(valid_smiles))
validity = len(valid_smiles) / NUM_GENERATE * 100
uniqueness = len(unique_smiles) / max(len(valid_smiles), 1) * 100

print(f'Generated: {NUM_GENERATE} molecules')
print(f'Valid:     {len(valid_smiles)} ({validity:.1f}%)  [{repaired_count} recovered via repair]')
print(f'Unique:    {len(unique_smiles)} ({uniqueness:.1f}%)')
print(f'\nSample generated SMILES:')
for smi in unique_smiles[:10]:
    print(f'  {smi}')

---
## 12. Optimized Candidate Molecules
Score generated molecules through all pipelines (toxicity, binding affinity, Lipinski) and rank the best candidates.

In [ ]:
# ─── Score Generated Molecules Through All Pipelines ───

def lipinski_score(smiles):
    """Return number of Lipinski Rule-of-5 violations."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 5
    violations = 0
    if Descriptors.MolWt(mol) > 500: violations += 1
    if Descriptors.MolLogP(mol) > 5: violations += 1
    if Descriptors.NumHDonors(mol) > 5: violations += 1
    if Descriptors.NumHAcceptors(mol) > 10: violations += 1
    return violations


def score_molecule(smiles, tox_model, egnn_model, device):
    """Score a molecule through toxicity and binding affinity predictors."""
    result = {'smiles': smiles}

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Molecular descriptors
    result['MW'] = Descriptors.MolWt(mol)
    result['LogP'] = Descriptors.MolLogP(mol)
    result['TPSA'] = Descriptors.TPSA(mol)
    result['HBD'] = Descriptors.NumHDonors(mol)
    result['HBA'] = Descriptors.NumHAcceptors(mol)
    result['lipinski_violations'] = lipinski_score(smiles)

    # Build graph
    graph = smiles_to_graph(smiles)
    if graph is None:
        return None

    # Toxicity prediction
    tox_model.eval()
    batch = Batch.from_data_list([graph]).to(device)
    with torch.no_grad():
        tox_logit = tox_model(batch).item()
        result['tox_prob'] = torch.sigmoid(torch.tensor(tox_logit)).item()

    # Binding affinity prediction
    egnn_model.eval()
    with torch.no_grad():
        aff_pred = egnn_model(batch.x, batch.edge_index, batch.batch).item()
        result['pred_affinity'] = aff_pred

    return result


# Score all valid generated molecules
scored_molecules = []
for smi in tqdm(unique_smiles, desc='Scoring molecules'):
    result = score_molecule(smi, tox_model, egnn_model, DEVICE)
    if result is not None:
        scored_molecules.append(result)

scored_df = pd.DataFrame(scored_molecules)
print(f'Successfully scored: {len(scored_df)} molecules')
scored_df.head()

In [ ]:
# ─── Rank and Select Optimized Candidates ───

# Composite optimization score:
# Higher affinity + lower toxicity + fewer Lipinski violations = better
scored_df['optimization_score'] = (
    scored_df['pred_affinity'] * 0.4                          # higher is better
    - scored_df['tox_prob'] * 3.0                             # lower is better
    - scored_df['lipinski_violations'] * 1.0                  # fewer violations
)

# Rank
scored_df = scored_df.sort_values('optimization_score', ascending=False).reset_index(drop=True)

print('\n=== TOP 15 OPTIMIZED CANDIDATE MOLECULES ===')
top_cols = ['smiles', 'MW', 'LogP', 'tox_prob', 'pred_affinity',
            'lipinski_violations', 'optimization_score']
display_df = scored_df[top_cols].head(15)
display_df.columns = ['SMILES', 'MW', 'LogP', 'Tox Prob', 'Pred Affinity',
                       'Lipinski Viol.', 'Opt. Score']
print(display_df.to_string(index=False))

In [ ]:
# ─── Visualize Optimized Candidates ───

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Toxicity vs Affinity scatter
sc = axes[0, 0].scatter(scored_df['pred_affinity'], scored_df['tox_prob'],
                         c=scored_df['optimization_score'], cmap='RdYlGn',
                         edgecolors='black', linewidths=0.3, alpha=0.7)
axes[0, 0].set_xlabel('Predicted Binding Affinity')
axes[0, 0].set_ylabel('Toxicity Probability')
axes[0, 0].set_title('Affinity vs Toxicity (color=Opt Score)')
plt.colorbar(sc, ax=axes[0, 0])

# 2. Optimization score distribution
axes[0, 1].hist(scored_df['optimization_score'], bins=30, color='seagreen', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=scored_df['optimization_score'].quantile(0.9), color='red',
                     linestyle='--', label='Top 10%')
axes[0, 1].set_xlabel('Optimization Score')
axes[0, 1].set_title('Optimization Score Distribution')
axes[0, 1].legend()

# 3. MW vs LogP (drug-likeness)
axes[1, 0].scatter(scored_df['MW'], scored_df['LogP'],
                    c=scored_df['lipinski_violations'], cmap='coolwarm',
                    edgecolors='black', linewidths=0.3, alpha=0.7)
axes[1, 0].axhline(y=5, color='red', linestyle='--', alpha=0.5)
axes[1, 0].axvline(x=500, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Molecular Weight')
axes[1, 0].set_ylabel('LogP')
axes[1, 0].set_title('Drug-likeness Space (Ro5 boundaries in red)')

# 4. Lipinski violations
scored_df['lipinski_violations'].value_counts().sort_index().plot.bar(
    ax=axes[1, 1], color='coral', edgecolor='black')
axes[1, 1].set_xlabel('Number of Lipinski Violations')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Lipinski Violations Distribution')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Draw Top Candidate Molecules ───

top_smiles = scored_df.head(8)['smiles'].tolist()
top_mols = [Chem.MolFromSmiles(s) for s in top_smiles if Chem.MolFromSmiles(s) is not None]

if top_mols:
    legends = [f'Score: {scored_df.iloc[i]["optimization_score"]:.2f}' for i in range(len(top_mols))]
    img = Draw.MolsToGridImage(top_mols, molsPerRow=4, subImgSize=(400, 300), legends=legends)
    display(img)
else:
    print('No valid molecules to draw. Try generating more or adjusting temperature.')

In [ ]:
# ─── Score Existing Drug Candidates from drug_cleaned ───

# Also evaluate top docking-scored molecules from existing libraries
top_docking = drug_cleaned.nsmallest(100, 'score')  # most negative = best docking

existing_scored = []
for _, row in tqdm(top_docking.iterrows(), total=len(top_docking), desc='Scoring existing drugs'):
    result = score_molecule(row['SMILES'], tox_model, egnn_model, DEVICE)
    if result is not None:
        result['docking_score'] = row['score']
        result['target'] = row['target']
        existing_scored.append(result)

existing_df = pd.DataFrame(existing_scored)
existing_df['optimization_score'] = (
    existing_df['pred_affinity'] * 0.4
    - existing_df['tox_prob'] * 3.0
    - existing_df['lipinski_violations'] * 1.0
)
existing_df = existing_df.sort_values('optimization_score', ascending=False)

print('\n=== TOP 10 EXISTING DRUG CANDIDATES (from docking library) ===')
print(existing_df[['smiles', 'target', 'docking_score', 'tox_prob',
                    'pred_affinity', 'lipinski_violations', 'optimization_score']].head(10).to_string(index=False))

In [ ]:
# ─── Export Optimized Candidates ───

# Save generated candidates
scored_df.to_csv('generated_candidates.csv', index=False)
print(f'Saved {len(scored_df)} generated candidates to generated_candidates.csv')

# Save existing library ranked candidates
existing_df.to_csv('existing_candidates_ranked.csv', index=False)
print(f'Saved {len(existing_df)} existing ranked candidates to existing_candidates_ranked.csv')

# Save model checkpoints
torch.save({
    'foundation_model': foundation_model.state_dict(),
    'diffusion_engine': diffusion_engine.state_dict(),
    'smiles_decoder': smiles_decoder.state_dict(),
    'egnn_model': egnn_model.state_dict(),
    'tox_model': tox_model.state_dict(),
}, 'drug_discovery_models.pt')
print('All model checkpoints saved to drug_discovery_models.pt')

---
## 13. Pipeline Summary

| Component | Model | Status |
|-----------|-------|--------|
| **Multimodal Embeddings** | Morgan FP (2048) + PhysChem Descriptors (10) | ✅ |
| **Transfer Learning** | Autoencoder Foundation Model (pre-trained) | ✅ |
| **Activity & ADMET Scoring** | Tox21 + ClinTox GNN classifier | ✅ |
| **Core Neural Architecture** | GCN Backbone (4-layer, residual) | ✅ |
| **Graph Diffusion Engine** | DiffRM (100-step, linear schedule) | ✅ |
| **Equivariant GNN** | EGNN w/ attention (binding affinity) | ✅ |
| **De Novo SMILES Generation** | Diffusion → GRU Decoder | ✅ |
| **Binding Affinity Predictor** | EGNN regression | ✅ |
| **Optimization & Ranking** | Multi-objective scoring | ✅ |